# Job scam screener on a GPU
Runtime > Change runtime type > **T4 GPU** first. Upload `laya-job-scam-screener.zip` and, for real numbers, `fake_job_postings.csv` (Kaggle: Real / Fake Job Posting Prediction, check its license).

In [4]:
!pip install -q -U laya pyyaml
import os; os.environ['USE_TF']='0'
from google.colab import files
up = files.upload()   # pick the zip (and the csv)

Saving laya-job-scam-screener.zip to laya-job-scam-screener.zip
Saving fake_job_postings.csv to fake_job_postings (1).csv


In [5]:
!unzip -q -o laya-job-scam-screener.zip
%cd laya-job-scam-screener
!mv ../fake_job_postings.csv data/ 2>/dev/null; ls data

/content/laya-job-scam-screener
fake_job_postings.csv  sample_postings.jsonl


In [7]:
# 1) which questions carry signal, and remove Laya's yes-bias (first run downloads the model, then it is fast)
!python scripts/diagnose.py --input data/fake_job_postings.csv --limit 600 --engine laya

600 labelled postings: 590 genuine, 10 scam
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1311.54it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
/usr/local/lib/python3.13/dist-packages/laya/router.py:260: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, device=self.device, token=self.token, subfolder=sub)
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1167.48it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Reconstructing 

In [8]:
# 2) fresh postings the fit never saw: measure honestly (rows 600-1200 -> save as a second file)
import pandas as pd
df = pd.read_csv('data/fake_job_postings.csv')
df.iloc[600:1800].to_csv('data/holdout.csv', index=False)
!python scripts/evaluate.py --input data/holdout.csv --calibration outputs/calibration.yaml

1200 labelled postings, 28 scam (2.3%)
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1357.29it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
/usr/local/lib/python3.13/dist-packages/laya/router.py:260: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, device=self.device, token=self.token, subfolder=sub)
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1343.38it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Reconstructing (inco

In [9]:
# 3) the demo run (timing on the GPU is what the dashboard will show)
!python run_pipeline.py --input data/holdout.csv --limit 500 --calibration outputs/calibration.yaml --out outputs/demo

Loaded 500 postings. Loading engine…
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1117.59it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
/usr/local/lib/python3.13/dist-packages/laya/router.py:260: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, device=self.device, token=self.token, subfolder=sub)
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1320.46it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Reconstructing (incomp

In [10]:
!cd outputs && zip -qr demo.zip demo
files.download('outputs/demo.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Laya vs Jev, both on a GPU (fair speed comparison)

Laya has no separate cloud API - it's self-hosted, so this Colab GPU is the fair stand-in for "Laya
on a server," comparable to Jev running on TypeSafe's own servers via OpenRouter. Running both from
here means neither side is handicapped by a laptop's CPU.

In [11]:
from getpass import getpass
import os
# openrouter.ai/settings/keys - no TypeSafe waitlist needed. Add a few dollars of credit first;
# there's no free tier. The key is not echoed or saved to the notebook file.
os.environ['OPENROUTER_API_KEY'] = getpass('OpenRouter API key: ')

OpenRouter API key: ··········


In [12]:
# Same postings, same checklist, both engines on this GPU. Keep --limit modest (20-50):
# Jev is billed per call, and this is meant to show HOW the two answer, not a full benchmark.
# For real accuracy numbers, run scripts/evaluate.py on each engine separately over hundreds of rows
# (see the two cells below this one).
!python scripts/compare.py --input data/fake_job_postings.csv --limit 30
from IPython.display import IFrame
IFrame('outputs/compare/compare.html', width='100%', height=700)

Comparing 30 postings on Laya and Jev. Jev is called live over the network, so this is illustrative, not a benchmark - see the module docstring.

-- Laya --
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1489.24it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
/usr/local/lib/python3.13/dist-packages/laya/router.py:260: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, device=self.device, token=self.token, subfolder=sub)
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1275.64it/s]
Download complete: :           |  0.00B            
Download complete: :

In [13]:
from IPython.display import HTML
HTML(open('outputs/compare/compare.html').read())

### Real accuracy numbers for each engine (run separately, both from this GPU)

Same idea as `diagnose.py` above, but for Jev. Compares AUROC on the SAME held-out postings.

In [14]:
!python scripts/evaluate.py --input data/holdout.csv --engine laya

1200 labelled postings, 28 scam (2.3%)
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1224.69it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
/usr/local/lib/python3.13/dist-packages/laya/router.py:260: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, device=self.device, token=self.token, subfolder=sub)
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1050.20it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Reconstructing (inco

In [15]:
import yaml
cfg = yaml.safe_load(open('config.yaml'))
cfg['engine']['http'] = {**cfg['engine'].get('http', {}), **cfg['jev_preset']}
cfg['engine']['type'] = 'http'
yaml.safe_dump(cfg, open('config.jev.yaml', 'w'), sort_keys=False)
!python scripts/evaluate.py --input data/holdout.csv --engine http --config config.jev.yaml

1200 labelled postings, 28 scam (2.3%)
timing: {'engine_seconds': 46.202, 'postings': 1200, 'ms_per_posting': 38.5, 'postings_per_second': 25.97, 'batch_size': 16}

At config settings (T=1.0): AUROC=0.601
  threshold check=0.3: {'precision': 0.04, 'recall': 0.464, 'f1': 0.073, 'tp': 13, 'fp': 316, 'fn': 15, 'tn': 856}
  threshold scam=0.6: {'precision': 0.208, 'recall': 0.179, 'f1': 0.192, 'tp': 5, 'fp': 19, 'fn': 23, 'tn': 1153}
Rules only: AUROC=0.523   (does Laya add anything on your data?)


In [16]:
!python scripts/evaluate.py --input data/holdout.csv --engine laya --calibration outputs/calibration.yaml

1200 labelled postings, 28 scam (2.3%)
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1147.62it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
/usr/local/lib/python3.13/dist-packages/laya/router.py:260: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, device=self.device, token=self.token, subfolder=sub)
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1224.19it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Reconstructing (inco